# Notebook 06: Predictive Performance and Threshold Selection

## Purpose

Evaluate logistic-regression and EBM out-of-fold probabilities under both targets, estimate paired bootstrap uncertainty, assess calibration, and select fold-specific classification thresholds within training data for 70%, 80%, and 90% sensitivity targets.

## Inputs

- Labelled complete-case data and fixed folds
- Logistic and EBM out-of-fold probabilities and saved model templates
- Metadata from Notebooks 04--05

## Outputs

- Performance metrics and paired bootstrap summaries
- Calibration-curve data and figures
- Fold-specific thresholds and thresholded out-of-fold predictions
- `data/processed/performance_and_thresholds_metadata.json`

## Dependencies

Run Notebooks 01--05 first. Notebook 07 consumes thresholded predictions; Notebooks 08--09 consume aggregate performance outputs.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## Evaluation design

### Probability performance

For each model–target combination, the notebook evaluates the complete vector of out-of-fold probabilities. Every participant's probability came from a model that did not use that participant for fitting.

The paired bootstrap resamples participants once per replicate and applies the same resampled indices to:

- both model classes;
- both target definitions.

This preserves the paired comparison structure. The bootstrap treats the completed out-of-fold prediction vectors as the evaluated predictions; it does not refit every model inside every bootstrap replicate.

### Threshold selection

For each outer held-out fold:

1. remove that fold completely;
2. create three inner folds among the remaining participants, stratified by the four-category joint label;
3. generate inner out-of-fold training probabilities using clones of the saved primary model specifications;
4. choose the highest threshold whose inner-training sensitivity reaches at least the requested target;
5. apply that threshold to the original outer-fold out-of-fold probabilities.

This nested procedure prevents the held-out fold's outcomes from influencing its threshold.

The primary operating point targets approximately 80% training sensitivity. The 70% and 90% operating points are retained for sensitivity analysis.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

try:
    import interpret
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "InterpretML is required to reload and clone the saved EBM models. "
        'Install the minimal package with: python -m pip install "interpret-core==0.7.8"'
    ) from exc

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 350)
pd.set_option("display.width", 200)


## 2. Fixed configuration

In [ ]:
RANDOM_STATE = 26
N_OUTER_SPLITS = 5
INNER_THRESHOLD_SPLITS = 3

N_BOOTSTRAP = int(
    os.environ.get(
        "PERFORMANCE_N_BOOTSTRAP",
        "1000",
    )
)

BOOTSTRAP_PROGRESS_EVERY = max(
    1,
    min(50, N_BOOTSTRAP),
)

CONFIDENCE_LEVEL = 0.95
CI_LOWER_QUANTILE = (
    1 - CONFIDENCE_LEVEL
) / 2
CI_UPPER_QUANTILE = (
    1 + CONFIDENCE_LEVEL
) / 2

CALIBRATION_BINS = 10
PROBABILITY_CLIP = 1e-6

TARGET_SENSITIVITY_LEVELS = [
    0.70,
    0.80,
    0.90,
]

PRIMARY_TARGET_SENSITIVITY = 0.80

MODEL_NAMES = [
    "logistic",
    "ebm",
]

MODEL_DISPLAY_NAMES = {
    "logistic": "Logistic regression",
    "ebm": "Explainable Boosting Machine",
}

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior reported clinician diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "Current HbA1c at least 6.5%"
    ),
}

TARGET_SHORT_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "HbA1c ≥ 6.5%"
    ),
}

JOINT_LABEL_ORDER = [
    "D0_H0",
    "D0_H1",
    "D1_H0",
    "D1_H1",
]

PROBABILITY_COLUMNS = {
    (
        "logistic",
        "self_reported_prior_diagnosis",
    ): (
        "logistic_oof_probability_prior_diagnosis"
    ),
    (
        "logistic",
        "current_hba1c_ge_6_5",
    ): (
        "logistic_oof_probability_hba1c_ge_6_5"
    ),
    (
        "ebm",
        "self_reported_prior_diagnosis",
    ): (
        "ebm_oof_probability_prior_diagnosis"
    ),
    (
        "ebm",
        "current_hba1c_ge_6_5",
    ): (
        "ebm_oof_probability_hba1c_ge_6_5"
    ),
}

METRIC_DISPLAY_NAMES = {
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
    "target_prevalence": (
        "Target prevalence / PR reference"
    ),
    "brier_score": "Brier score",
    "brier_skill_score": (
        "Brier skill score"
    ),
    "calibration_intercept": (
        "Calibration intercept"
    ),
    "calibration_slope": (
        "Calibration slope"
    ),
    "sensitivity_at_0_5": (
        "Sensitivity at threshold 0.5"
    ),
    "specificity_at_0_5": (
        "Specificity at threshold 0.5"
    ),
}

METRIC_ORDER = list(
    METRIC_DISPLAY_NAMES
)

print("Performance bootstrap replicates:", N_BOOTSTRAP)
print(
    "Inner threshold-selection folds:",
    INNER_THRESHOLD_SPLITS,
)


### Interpretation of Brier skill score

For each target,

$$
\text{Brier skill score}
=
1-
\frac{\text{model Brier score}}
     {\text{Brier score from a constant prevalence prediction}}.
$$

A value above zero means that the model improves on the target-specific constant-prevalence prediction. This scaling provides useful prevalence context, but it does not make different targets fully interchangeable.

## 3. Project paths

In [ ]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = (
    PROJECT_DIR / "data" / "processed"
)
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LABELLED_DATA_PATH = (
    PROCESSED_DIR
    / "nhanes_diabetes_complete_case_labeled.csv"
)

FOLD_ASSIGNMENT_PATH = (
    PROCESSED_DIR
    / "primary_cv_fold_assignments.csv"
)

LOGISTIC_OOF_PATH = (
    PROCESSED_DIR
    / "logistic_oof_predictions.csv"
)

EBM_OOF_PATH = (
    PROCESSED_DIR
    / "ebm_oof_predictions.csv"
)

LOGISTIC_METADATA_PATH = (
    PROCESSED_DIR
    / "logistic_regression_metadata.json"
)

EBM_METADATA_PATH = (
    PROCESSED_DIR
    / "ebm_analysis_metadata.json"
)

LOGISTIC_MODEL_PATHS = {
    "self_reported_prior_diagnosis": (
        MODEL_DIR
        / "logistic_self_reported_prior_diagnosis.joblib"
    ),
    "current_hba1c_ge_6_5": (
        MODEL_DIR
        / "logistic_current_hba1c_ge_6_5.joblib"
    ),
}

EBM_MODEL_PATHS = {
    "self_reported_prior_diagnosis": (
        MODEL_DIR
        / "ebm_self_reported_prior_diagnosis.joblib"
    ),
    "current_hba1c_ge_6_5": (
        MODEL_DIR
        / "ebm_current_hba1c_ge_6_5.joblib"
    ),
}

PERFORMANCE_METADATA_PATH = (
    PROCESSED_DIR
    / "performance_and_thresholds_metadata.json"
)

THRESHOLDED_LONG_PATH = (
    PROCESSED_DIR
    / "thresholded_oof_predictions_long.csv"
)

THRESHOLDED_WIDE_PATH = (
    PROCESSED_DIR
    / "thresholded_oof_predictions_wide.csv"
)

PRIMARY_THRESHOLD_PATH = (
    PROCESSED_DIR
    / "primary_80_sensitivity_thresholded_oof_predictions.csv"
)

required_paths = [
    LABELLED_DATA_PATH,
    FOLD_ASSIGNMENT_PATH,
    LOGISTIC_OOF_PATH,
    EBM_OOF_PATH,
    LOGISTIC_METADATA_PATH,
    EBM_METADATA_PATH,
    *LOGISTIC_MODEL_PATHS.values(),
    *EBM_MODEL_PATHS.values(),
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(
        f"- {path}"
        for path in missing_paths
    )

    raise FileNotFoundError(
        "Notebook 06 requires completed outputs from Notebooks 04 and 05. "
        "The following files are missing:\n"
        f"{missing_text}"
    )

print("Project directory:", PROJECT_DIR)
print("Logistic OOF predictions:", LOGISTIC_OOF_PATH)
print("EBM OOF predictions:", EBM_OOF_PATH)


## 4. Load data, predictions, metadata, and model templates

In [ ]:
data = pd.read_csv(
    LABELLED_DATA_PATH
)

fold_assignments = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

logistic_oof = pd.read_csv(
    LOGISTIC_OOF_PATH
)

ebm_oof = pd.read_csv(
    EBM_OOF_PATH
)

with LOGISTIC_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    logistic_metadata = json.load(file)

with EBM_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    ebm_metadata = json.load(file)

model_templates = {
    "logistic": {
        target: joblib.load(path)
        for target, path
        in LOGISTIC_MODEL_PATHS.items()
    },
    "ebm": {
        target: joblib.load(path)
        for target, path
        in EBM_MODEL_PATHS.items()
    },
}

PREDICTOR_COLUMNS = list(
    logistic_metadata[
        "predictor_columns"
    ]
)

CONTINUOUS_PREDICTORS = list(
    logistic_metadata[
        "continuous_predictors"
    ]
)

CATEGORICAL_PREDICTORS = list(
    logistic_metadata[
        "categorical_predictors"
    ]
)

print("Loaded analytic data:", data.shape)
print("Loaded logistic predictions:", logistic_oof.shape)
print("Loaded EBM predictions:", ebm_oof.shape)
print("Predictors:", PREDICTOR_COLUMNS)


## 5. Cross-notebook validation

In [ ]:
def sorted_by_id(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    result = dataframe.copy()
    result["id"] = pd.to_numeric(
        result["id"],
        errors="raise",
    ).astype("Int64")

    return (
        result
        .sort_values("id")
        .reset_index(drop=True)
    )


data = sorted_by_id(data)
fold_assignments = sorted_by_id(
    fold_assignments
)
logistic_oof = sorted_by_id(
    logistic_oof
)
ebm_oof = sorted_by_id(
    ebm_oof
)

required_data_columns = set(
    [
        "id",
        "joint_label_code",
        "confirmed_current_pregnancy",
        "primary_sample_eligible",
    ]
    + PREDICTOR_COLUMNS
    + TARGET_COLUMNS
)

missing_data_columns = (
    required_data_columns
    .difference(data.columns)
)

if missing_data_columns:
    raise KeyError(
        "The labelled data are missing columns: "
        f"{sorted(missing_data_columns)}"
    )

required_fold_columns = {
    "id",
    "cv_fold",
    "joint_label_code",
    *TARGET_COLUMNS,
}

missing_fold_columns = (
    required_fold_columns
    .difference(fold_assignments.columns)
)

if missing_fold_columns:
    raise KeyError(
        "The fold file is missing columns: "
        f"{sorted(missing_fold_columns)}"
    )

for target in TARGET_COLUMNS:
    data[target] = pd.to_numeric(
        data[target],
        errors="raise",
    ).astype(int)

data[
    "confirmed_current_pregnancy"
] = pd.to_numeric(
    data["confirmed_current_pregnancy"],
    errors="raise",
).astype(int)

data[
    "primary_sample_eligible"
] = pd.to_numeric(
    data["primary_sample_eligible"],
    errors="raise",
).astype(int)

fold_assignments["cv_fold"] = (
    pd.to_numeric(
        fold_assignments["cv_fold"],
        errors="raise",
    ).astype(int)
)

for dataframe in [
    fold_assignments,
    logistic_oof,
    ebm_oof,
]:
    for target in TARGET_COLUMNS:
        dataframe[target] = (
            pd.to_numeric(
                dataframe[target],
                errors="raise",
            ).astype(int)
        )

for variable in CONTINUOUS_PREDICTORS:
    data[variable] = pd.to_numeric(
        data[variable],
        errors="raise",
    ).astype(float)

for variable in CATEGORICAL_PREDICTORS:
    data[variable] = (
        data[variable]
        .astype("string")
    )

data["joint_label_code"] = (
    data["joint_label_code"]
    .astype("string")
)

fold_assignments[
    "joint_label_code"
] = (
    fold_assignments[
        "joint_label_code"
    ].astype("string")
)

if not data["id"].is_unique:
    raise ValueError(
        "The analytic data contain duplicate IDs."
    )

for name, dataframe in {
    "fold assignments": fold_assignments,
    "logistic OOF predictions": logistic_oof,
    "EBM OOF predictions": ebm_oof,
}.items():
    if not dataframe["id"].is_unique:
        raise ValueError(
            f"The {name} contain duplicate IDs."
        )

base_ids = set(
    data["id"].astype(int)
)

for name, dataframe in {
    "fold assignments": fold_assignments,
    "logistic OOF predictions": logistic_oof,
    "EBM OOF predictions": ebm_oof,
}.items():
    if set(
        dataframe["id"].astype(int)
    ) != base_ids:
        raise ValueError(
            f"The {name} belong to a different analytic sample."
        )

comparison_columns = [
    "joint_label_code",
    *TARGET_COLUMNS,
]

for name, dataframe in {
    "fold assignments": fold_assignments,
    "logistic OOF predictions": logistic_oof,
    "EBM OOF predictions": ebm_oof,
}.items():
    for column in comparison_columns:
        if not (
            data[column]
            .astype(str)
            .eq(
                dataframe[column]
                .astype(str)
            )
            .all()
        ):
            raise ValueError(
                f"The {name} disagree with the analytic data in {column}."
            )

if data[
    "confirmed_current_pregnancy"
].ne(0).any():
    raise ValueError(
        "The analytic data contain a confirmed current pregnancy."
    )

if not data[
    "primary_sample_eligible"
].eq(1).all():
    raise ValueError(
        "The analytic data contain an ineligible participant."
    )

if set(
    fold_assignments["cv_fold"]
) != set(
    range(1, N_OUTER_SPLITS + 1)
):
    raise ValueError(
        "The fixed assignment does not contain exactly five folds."
    )

data = data.merge(
    fold_assignments[
        ["id", "cv_fold"]
    ],
    on="id",
    how="left",
    validate="one_to_one",
)

logistic_probability_columns = [
    PROBABILITY_COLUMNS[
        ("logistic", target)
    ]
    for target in TARGET_COLUMNS
]

ebm_probability_columns = [
    PROBABILITY_COLUMNS[
        ("ebm", target)
    ]
    for target in TARGET_COLUMNS
]

missing_logistic_probability_columns = set(
    logistic_probability_columns
).difference(
    logistic_oof.columns
)

missing_ebm_probability_columns = set(
    ebm_probability_columns
).difference(
    ebm_oof.columns
)

if missing_logistic_probability_columns:
    raise KeyError(
        "Missing logistic probability columns: "
        f"{sorted(missing_logistic_probability_columns)}"
    )

if missing_ebm_probability_columns:
    raise KeyError(
        "Missing EBM probability columns: "
        f"{sorted(missing_ebm_probability_columns)}"
    )

analysis = (
    data
    .merge(
        logistic_oof[
            [
                "id",
                *logistic_probability_columns,
            ]
        ],
        on="id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        ebm_oof[
            [
                "id",
                *ebm_probability_columns,
            ]
        ],
        on="id",
        how="left",
        validate="one_to_one",
    )
)

for (
    model_name,
    target
), probability_column in PROBABILITY_COLUMNS.items():
    analysis[
        probability_column
    ] = pd.to_numeric(
        analysis[probability_column],
        errors="raise",
    ).astype(float)

    if analysis[
        probability_column
    ].isna().any():
        raise ValueError(
            f"Missing probabilities in {probability_column}."
        )

    if not analysis[
        probability_column
    ].between(0, 1).all():
        raise ValueError(
            f"Probabilities outside [0, 1] in {probability_column}."
        )

analysis_id_hash = hashlib.sha256(
    ",".join(
        analysis["id"]
        .astype(str)
        .tolist()
    ).encode("utf-8")
).hexdigest()

if (
    logistic_metadata[
        "analysis_id_sha256"
    ]
    != analysis_id_hash
):
    raise ValueError(
        "Notebook 04 metadata belong to a different participant sample."
    )

if (
    ebm_metadata[
        "analysis_id_sha256"
    ]
    != analysis_id_hash
):
    raise ValueError(
        "Notebook 05 metadata belong to a different participant sample."
    )

if (
    logistic_metadata[
        "predictor_columns"
    ]
    != ebm_metadata[
        "predictor_columns"
    ]
):
    raise ValueError(
        "The logistic and EBM predictor sets differ."
    )

if (
    logistic_metadata[
        "target_columns"
    ]
    != ebm_metadata[
        "target_columns"
    ]
):
    raise ValueError(
        "The logistic and EBM target orders differ."
    )

if (
    logistic_metadata["random_state"]
    != RANDOM_STATE
    or ebm_metadata["random_state"]
    != RANDOM_STATE
):
    raise ValueError(
        "An upstream notebook used a different random seed."
    )

fold_joint_balance = (
    analysis
    .groupby(
        ["cv_fold", "joint_label_code"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=range(
            1,
            N_OUTER_SPLITS + 1,
        ),
        columns=JOINT_LABEL_ORDER,
        fill_value=0,
    )
)

if (
    fold_joint_balance == 0
).any().any():
    raise ValueError(
        "At least one outer fold is missing a joint-label group."
    )

validation_summary = {
    "analytic_sample_n": int(
        len(analysis)
    ),
    "analysis_id_sha256": (
        analysis_id_hash
    ),
    "same_sample_as_logistic": True,
    "same_sample_as_ebm": True,
    "fixed_five_folds_present": True,
    "probability_columns_checked": (
        list(PROBABILITY_COLUMNS.values())
    ),
    "missing_probabilities": 0,
    "all_probabilities_in_unit_interval": True,
}

print("Cross-notebook validation passed.")
validation_summary


## Section 9 — Predictive performance

## 6. Metric helpers

In [ ]:
def safe_logit(
    probability: np.ndarray,
) -> np.ndarray:
    clipped = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1 - PROBABILITY_CLIP,
    )

    return np.log(
        clipped
        / (1 - clipped)
    )


def estimate_calibration_intercept_slope(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=float,
    )

    linear_predictor = safe_logit(
        probability
    )

    def objective(parameters):
        intercept, slope = parameters
        eta = (
            intercept
            + slope * linear_predictor
        )

        return float(
            np.sum(
                np.logaddexp(
                    0,
                    eta,
                )
                - y * eta
            )
        )

    def gradient(parameters):
        intercept, slope = parameters
        eta = (
            intercept
            + slope * linear_predictor
        )

        fitted = np.where(
            eta >= 0,
            1 / (
                1 + np.exp(-eta)
            ),
            np.exp(eta)
            / (
                1 + np.exp(eta)
            ),
        )

        residual = fitted - y

        return np.array(
            [
                residual.sum(),
                np.sum(
                    residual
                    * linear_predictor
                ),
            ],
            dtype=float,
        )

    result = minimize(
        objective,
        x0=np.array(
            [0.0, 1.0],
            dtype=float,
        ),
        jac=gradient,
        method="BFGS",
    )

    if not result.success:
        return {
            "calibration_intercept": np.nan,
            "calibration_slope": np.nan,
            "calibration_fit_success": False,
        }

    return {
        "calibration_intercept": float(
            result.x[0]
        ),
        "calibration_slope": float(
            result.x[1]
        ),
        "calibration_fit_success": True,
    }


def binary_classification_counts(
    y_true: np.ndarray,
    predicted_class: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    predicted = np.asarray(
        predicted_class,
        dtype=int,
    )

    true_positive = int(
        np.sum(
            (y == 1)
            & (predicted == 1)
        )
    )

    false_negative = int(
        np.sum(
            (y == 1)
            & (predicted == 0)
        )
    )

    true_negative = int(
        np.sum(
            (y == 0)
            & (predicted == 0)
        )
    )

    false_positive = int(
        np.sum(
            (y == 0)
            & (predicted == 1)
        )
    )

    return {
        "true_positive": true_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "false_positive": false_positive,
    }


def safe_ratio(
    numerator: float,
    denominator: float,
) -> float:
    if denominator == 0:
        return np.nan

    return float(
        numerator / denominator
    )


def probability_performance_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    p = np.asarray(
        probability,
        dtype=float,
    )

    if np.unique(y).size != 2:
        raise ValueError(
            "Both outcome classes are required for performance metrics."
        )

    prevalence = float(
        y.mean()
    )

    brier = float(
        brier_score_loss(
            y,
            p,
        )
    )

    null_brier = float(
        np.mean(
            (
                y
                - prevalence
            ) ** 2
        )
    )

    brier_skill = (
        1
        - brier / null_brier
        if null_brier > 0
        else np.nan
    )

    predicted_at_half = (
        p >= 0.5
    ).astype(int)

    counts = (
        binary_classification_counts(
            y,
            predicted_at_half,
        )
    )

    sensitivity = safe_ratio(
        counts["true_positive"],
        (
            counts["true_positive"]
            + counts["false_negative"]
        ),
    )

    specificity = safe_ratio(
        counts["true_negative"],
        (
            counts["true_negative"]
            + counts["false_positive"]
        ),
    )

    calibration = (
        estimate_calibration_intercept_slope(
            y,
            p,
        )
    )

    return {
        "roc_auc": float(
            roc_auc_score(
                y,
                p,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y,
                p,
            )
        ),
        "target_prevalence": (
            prevalence
        ),
        "brier_score": brier,
        "brier_skill_score": float(
            brier_skill
        ),
        "calibration_intercept": (
            calibration[
                "calibration_intercept"
            ]
        ),
        "calibration_slope": (
            calibration[
                "calibration_slope"
            ]
        ),
        "calibration_fit_success": bool(
            calibration[
                "calibration_fit_success"
            ]
        ),
        "sensitivity_at_0_5": (
            sensitivity
        ),
        "specificity_at_0_5": (
            specificity
        ),
    }


## 7. Calculate point estimates for all four combinations

In [ ]:
performance_point_rows = []

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        y = analysis[
            target
        ].to_numpy(
            dtype=int
        )

        probability_column = (
            PROBABILITY_COLUMNS[
                (model_name, target)
            ]
        )

        probability = analysis[
            probability_column
        ].to_numpy(
            dtype=float
        )

        metrics = (
            probability_performance_metrics(
                y,
                probability,
            )
        )

        for metric in METRIC_ORDER:
            performance_point_rows.append(
                {
                    "model": model_name,
                    "model_display_name": (
                        MODEL_DISPLAY_NAMES[
                            model_name
                        ]
                    ),
                    "target": target,
                    "target_display_name": (
                        TARGET_DISPLAY_NAMES[
                            target
                        ]
                    ),
                    "metric": metric,
                    "metric_display_name": (
                        METRIC_DISPLAY_NAMES[
                            metric
                        ]
                    ),
                    "estimate": float(
                        metrics[metric]
                    ),
                    "evaluation_n": int(
                        len(y)
                    ),
                    "positive_n": int(
                        y.sum()
                    ),
                    "negative_n": int(
                        (1 - y).sum()
                    ),
                    "calibration_fit_success": bool(
                        metrics[
                            "calibration_fit_success"
                        ]
                    ),
                }
            )

performance_point_estimates = (
    pd.DataFrame(
        performance_point_rows
    )
)

performance_point_estimates.to_csv(
    TABLE_DIR
    / "performance_point_estimates_long.csv",
    index=False,
)

performance_point_estimates


### Interpretation of sensitivity and specificity in Table 3

The sensitivity and specificity values in the primary probability-performance table use a conventional threshold of 0.5 and are labelled explicitly as such.

They are not the only operating-point results. The threshold section below creates the prespecified 70%, 80%, and 90% sensitivity operating points without using held-out outcomes.

## 8. Participant-level paired bootstrap

In [ ]:
bootstrap_rng = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_rows = []
bootstrap_start_time = time.time()
calibration_failure_count = 0

for replicate in range(
    1,
    N_BOOTSTRAP + 1,
):
    sampled_indices = (
        bootstrap_rng.integers(
            low=0,
            high=len(analysis),
            size=len(analysis),
        )
    )

    sampled_data = (
        analysis
        .iloc[sampled_indices]
    )

    for model_name in MODEL_NAMES:
        for target in TARGET_COLUMNS:
            y = sampled_data[
                target
            ].to_numpy(
                dtype=int
            )

            probability_column = (
                PROBABILITY_COLUMNS[
                    (
                        model_name,
                        target,
                    )
                ]
            )

            probability = (
                sampled_data[
                    probability_column
                ].to_numpy(
                    dtype=float
                )
            )

            metrics = (
                probability_performance_metrics(
                    y,
                    probability,
                )
            )

            if not metrics[
                "calibration_fit_success"
            ]:
                calibration_failure_count += 1

            for metric in METRIC_ORDER:
                bootstrap_rows.append(
                    {
                        "bootstrap_replicate": (
                            replicate
                        ),
                        "model": model_name,
                        "target": target,
                        "metric": metric,
                        "estimate": (
                            metrics[metric]
                        ),
                    }
                )

    if (
        replicate
        % BOOTSTRAP_PROGRESS_EVERY
        == 0
        or replicate == N_BOOTSTRAP
    ):
        elapsed_minutes = (
            time.time()
            - bootstrap_start_time
        ) / 60

        print(
            f"Completed {replicate:,}/{N_BOOTSTRAP:,} "
            f"bootstrap replicates "
            f"({elapsed_minutes:.1f} minutes elapsed)."
        )

bootstrap_performance = pd.DataFrame(
    bootstrap_rows
)

expected_bootstrap_rows = (
    N_BOOTSTRAP
    * len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * len(METRIC_ORDER)
)

if (
    len(bootstrap_performance)
    != expected_bootstrap_rows
):
    raise ValueError(
        "Unexpected number of bootstrap metric rows."
    )

bootstrap_performance.to_csv(
    TABLE_DIR
    / "performance_bootstrap_metrics_long.csv",
    index=False,
)

print(
    "Calibration fits that returned missing estimates:",
    calibration_failure_count,
)


## 9. Bootstrap confidence intervals for each metric

In [ ]:
performance_bootstrap_summary = (
    bootstrap_performance
    .groupby(
        [
            "model",
            "target",
            "metric",
        ],
        as_index=False,
    )["estimate"]
    .agg(
        bootstrap_mean="mean",
        bootstrap_standard_error="std",
        ci_lower=(
            lambda values:
            values.dropna().quantile(
                CI_LOWER_QUANTILE
            )
        ),
        ci_upper=(
            lambda values:
            values.dropna().quantile(
                CI_UPPER_QUANTILE
            )
        ),
        successful_bootstrap_replicates="count",
    )
)

performance_summary = (
    performance_point_estimates
    .merge(
        performance_bootstrap_summary,
        on=[
            "model",
            "target",
            "metric",
        ],
        how="left",
        validate="one_to_one",
    )
)

performance_summary[
    "bootstrap_replicates_requested"
] = N_BOOTSTRAP

performance_summary.to_csv(
    TABLE_DIR
    / "performance_metrics_with_bootstrap_intervals.csv",
    index=False,
)

performance_summary


## 10. Table 3: primary predictive performance

In [ ]:
def metric_digits(
    metric: str,
) -> int:
    if metric in {
        "calibration_intercept",
        "calibration_slope",
    }:
        return 3

    return 3


def format_estimate_interval(
    estimate: float,
    lower: float,
    upper: float,
    digits: int,
) -> str:
    if (
        pd.isna(estimate)
        or pd.isna(lower)
        or pd.isna(upper)
    ):
        return "Not estimable"

    return (
        f"{estimate:.{digits}f} "
        f"[{lower:.{digits}f}, "
        f"{upper:.{digits}f}]"
    )


table3_long = (
    performance_summary
    .copy()
)

table3_long[
    "estimate_95_ci"
] = table3_long.apply(
    lambda row:
    format_estimate_interval(
        row["estimate"],
        row["ci_lower"],
        row["ci_upper"],
        metric_digits(
            row["metric"]
        ),
    ),
    axis=1,
)

table3_wide = (
    table3_long
    .pivot(
        index=[
            "model",
            "model_display_name",
            "target",
            "target_display_name",
        ],
        columns="metric",
        values="estimate_95_ci",
    )
    .reset_index()
)

table3_wide.columns.name = None

ordered_table3_columns = [
    "model",
    "model_display_name",
    "target",
    "target_display_name",
    *METRIC_ORDER,
]

table3_wide = table3_wide[
    ordered_table3_columns
]

table3_long.to_csv(
    TABLE_DIR
    / "table3_primary_predictive_performance_long.csv",
    index=False,
)

table3_wide.to_csv(
    TABLE_DIR
    / "table3_primary_predictive_performance_wide.csv",
    index=False,
)

table3_wide


## Primary within-target model comparisons

## 11. EBM minus logistic-regression differences

These are the primary formal model comparisons because both models are evaluated against the same target on the same participants.

In [ ]:
point_model_difference_wide = (
    performance_point_estimates
    .pivot(
        index=[
            "target",
            "metric",
        ],
        columns="model",
        values="estimate",
    )
    .reset_index()
)

point_model_difference_wide[
    "difference_ebm_minus_logistic"
] = (
    point_model_difference_wide[
        "ebm"
    ]
    - point_model_difference_wide[
        "logistic"
    ]
)

bootstrap_model_difference_wide = (
    bootstrap_performance
    .pivot(
        index=[
            "bootstrap_replicate",
            "target",
            "metric",
        ],
        columns="model",
        values="estimate",
    )
    .reset_index()
)

bootstrap_model_difference_wide[
    "difference_ebm_minus_logistic"
] = (
    bootstrap_model_difference_wide[
        "ebm"
    ]
    - bootstrap_model_difference_wide[
        "logistic"
    ]
)

model_difference_intervals = (
    bootstrap_model_difference_wide
    .groupby(
        [
            "target",
            "metric",
        ],
        as_index=False,
    )[
        "difference_ebm_minus_logistic"
    ]
    .agg(
        bootstrap_mean_difference="mean",
        bootstrap_standard_error_difference="std",
        difference_ci_lower=(
            lambda values:
            values.dropna().quantile(
                CI_LOWER_QUANTILE
            )
        ),
        difference_ci_upper=(
            lambda values:
            values.dropna().quantile(
                CI_UPPER_QUANTILE
            )
        ),
        successful_bootstrap_replicates="count",
    )
)

within_target_model_differences = (
    point_model_difference_wide
    .merge(
        model_difference_intervals,
        on=[
            "target",
            "metric",
        ],
        how="left",
        validate="one_to_one",
    )
)

within_target_model_differences[
    "target_display_name"
] = (
    within_target_model_differences[
        "target"
    ].map(
        TARGET_DISPLAY_NAMES
    )
)

within_target_model_differences[
    "metric_display_name"
] = (
    within_target_model_differences[
        "metric"
    ].map(
        METRIC_DISPLAY_NAMES
    )
)

within_target_model_differences.to_csv(
    TABLE_DIR
    / "performance_within_target_ebm_minus_logistic.csv",
    index=False,
)

within_target_model_differences


### Direction of model differences

For ROC-AUC, PR-AUC, Brier skill score, calibration slope, sensitivity, and specificity, larger values are not always uniformly better without context, but positive EBM-minus-logistic differences indicate numerically larger EBM values.

For Brier score, lower values are better, so a negative EBM-minus-logistic difference favours EBM.

Calibration intercept is ideally near zero and calibration slope is ideally near one. Their signed differences should therefore be interpreted relative to those reference values rather than as a simple larger-is-better ranking.

## Descriptive cross-target comparisons

## 12. HbA1c-target minus prior-diagnosis-target differences

These comparisons are paired because both targets are observed for the same participants, but they remain descriptive because the outcomes define different tasks.

In [ ]:
point_cross_target_wide = (
    performance_point_estimates
    .pivot(
        index=[
            "model",
            "metric",
        ],
        columns="target",
        values="estimate",
    )
    .reset_index()
)

point_cross_target_wide[
    "difference_hba1c_minus_prior"
] = (
    point_cross_target_wide[
        "current_hba1c_ge_6_5"
    ]
    - point_cross_target_wide[
        "self_reported_prior_diagnosis"
    ]
)

bootstrap_cross_target_wide = (
    bootstrap_performance
    .pivot(
        index=[
            "bootstrap_replicate",
            "model",
            "metric",
        ],
        columns="target",
        values="estimate",
    )
    .reset_index()
)

bootstrap_cross_target_wide[
    "difference_hba1c_minus_prior"
] = (
    bootstrap_cross_target_wide[
        "current_hba1c_ge_6_5"
    ]
    - bootstrap_cross_target_wide[
        "self_reported_prior_diagnosis"
    ]
)

cross_target_intervals = (
    bootstrap_cross_target_wide
    .groupby(
        [
            "model",
            "metric",
        ],
        as_index=False,
    )[
        "difference_hba1c_minus_prior"
    ]
    .agg(
        bootstrap_mean_difference="mean",
        bootstrap_standard_error_difference="std",
        difference_ci_lower=(
            lambda values:
            values.dropna().quantile(
                CI_LOWER_QUANTILE
            )
        ),
        difference_ci_upper=(
            lambda values:
            values.dropna().quantile(
                CI_UPPER_QUANTILE
            )
        ),
        successful_bootstrap_replicates="count",
    )
)

cross_target_performance_differences = (
    point_cross_target_wide
    .merge(
        cross_target_intervals,
        on=[
            "model",
            "metric",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_target_performance_differences[
    "model_display_name"
] = (
    cross_target_performance_differences[
        "model"
    ].map(
        MODEL_DISPLAY_NAMES
    )
)

cross_target_performance_differences[
    "metric_display_name"
] = (
    cross_target_performance_differences[
        "metric"
    ].map(
        METRIC_DISPLAY_NAMES
    )
)

cross_target_performance_differences[
    "comparison_scope"
] = (
    "Descriptive cross-target comparison; "
    "the two outcomes define different prediction tasks"
)

cross_target_performance_differences.to_csv(
    TABLE_DIR
    / "performance_cross_target_hba1c_minus_prior.csv",
    index=False,
)

cross_target_performance_differences


### Restrictions on cross-target interpretation

- PR-AUC depends strongly on target prevalence.
- Raw Brier scores also depend on outcome prevalence and uncertainty.
- Brier skill score adds a target-specific prevalence reference but does not make the targets identical.
- A larger AUC for one outcome does not establish that it is the more clinically valid outcome.
- Cross-target differences describe how the same predictor information performs under two different operational labels.

## Calibration curves

## 13. Calculate quantile-binned calibration points

In [ ]:
calibration_rows = []

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        y = analysis[
            target
        ].to_numpy(
            dtype=int
        )

        probability = analysis[
            PROBABILITY_COLUMNS[
                (
                    model_name,
                    target,
                )
            ]
        ].to_numpy(
            dtype=float
        )

        observed_fraction, mean_probability = (
            calibration_curve(
                y,
                probability,
                n_bins=CALIBRATION_BINS,
                strategy="quantile",
            )
        )

        probability_bins = pd.qcut(
            probability,
            q=CALIBRATION_BINS,
            duplicates="drop",
        )

        bin_counts = (
            pd.Series(
                probability_bins
            )
            .value_counts(
                sort=False
            )
            .to_numpy()
        )

        if (
            len(bin_counts)
            != len(mean_probability)
        ):
            raise ValueError(
                "Calibration bin counts do not align with calibration points."
            )

        for bin_index, (
            predicted_mean,
            observed_mean,
            bin_n,
        ) in enumerate(
            zip(
                mean_probability,
                observed_fraction,
                bin_counts,
            ),
            start=1,
        ):
            calibration_rows.append(
                {
                    "model": model_name,
                    "model_display_name": (
                        MODEL_DISPLAY_NAMES[
                            model_name
                        ]
                    ),
                    "target": target,
                    "target_display_name": (
                        TARGET_DISPLAY_NAMES[
                            target
                        ]
                    ),
                    "calibration_bin": (
                        bin_index
                    ),
                    "bin_n": int(
                        bin_n
                    ),
                    "mean_predicted_probability": float(
                        predicted_mean
                    ),
                    "observed_outcome_fraction": float(
                        observed_mean
                    ),
                }
            )

calibration_points = pd.DataFrame(
    calibration_rows
)

calibration_points.to_csv(
    TABLE_DIR
    / "calibration_curve_points.csv",
    index=False,
)

calibration_points


## 14. Plot one calibration figure per target

In [ ]:
calibration_figure_paths = {}

for target in TARGET_COLUMNS:
    figure, axis = plt.subplots(
        figsize=(6.5, 6)
    )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration",
    )

    for model_name in MODEL_NAMES:
        target_model_data = (
            calibration_points
            .loc[
                (
                    calibration_points[
                        "target"
                    ]
                    == target
                )
                & (
                    calibration_points[
                        "model"
                    ]
                    == model_name
                )
            ]
            .sort_values(
                "mean_predicted_probability"
            )
        )

        axis.plot(
            target_model_data[
                "mean_predicted_probability"
            ],
            target_model_data[
                "observed_outcome_fraction"
            ],
            marker="o",
            label=(
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            ),
        )

    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)

    axis.set_xlabel(
        "Mean out-of-fold predicted probability"
    )

    axis.set_ylabel(
        "Observed outcome proportion"
    )

    axis.set_title(
        "Calibration: "
        f"{TARGET_SHORT_NAMES[target]}"
    )

    axis.legend()
    figure.tight_layout()

    filename_target = (
        "prior_diagnosis"
        if target
        == "self_reported_prior_diagnosis"
        else "hba1c_ge_6_5"
    )

    png_path = (
        FIGURE_DIR
        / (
            "performance_calibration_"
            f"{filename_target}.png"
        )
    )

    pdf_path = (
        FIGURE_DIR
        / (
            "performance_calibration_"
            f"{filename_target}.pdf"
        )
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    calibration_figure_paths[
        f"{filename_target}_png"
    ] = png_path

    calibration_figure_paths[
        f"{filename_target}_pdf"
    ] = pdf_path

calibration_figure_paths


## Section 10 — Classification thresholds

## 15. Prepare model inputs and verify that saved models are cloneable

In [ ]:
def prepare_predictor_matrix(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    X = dataframe[
        PREDICTOR_COLUMNS
    ].copy()

    for variable in CONTINUOUS_PREDICTORS:
        X[variable] = pd.to_numeric(
            X[variable],
            errors="raise",
        ).astype(float)

    for variable in CATEGORICAL_PREDICTORS:
        X[variable] = (
            X[variable]
            .astype(str)
        )

    return X


def fresh_estimator(
    template,
):
    try:
        return clone(template)
    except Exception as exc:
        raise RuntimeError(
            "A saved primary model could not be cloned for nested "
            "threshold selection. Rerun the corresponding modelling "
            "notebook with the current software environment."
        ) from exc


X_all = prepare_predictor_matrix(
    analysis
)

for model_name in MODEL_NAMES:
    for target in TARGET_COLUMNS:
        cloned_model = fresh_estimator(
            model_templates[
                model_name
            ][target]
        )

        if cloned_model is None:
            raise RuntimeError(
                "Unexpected empty model clone."
            )

print(
    "All four saved primary model specifications "
    "can be cloned."
)


## 16. Threshold-selection helpers

In [ ]:
def select_threshold_for_minimum_sensitivity(
    y_true: np.ndarray,
    probability: np.ndarray,
    target_sensitivity: float,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    p = np.asarray(
        probability,
        dtype=float,
    )

    positive_probabilities = (
        p[y == 1]
    )

    if (
        positive_probabilities.size
        == 0
    ):
        raise ValueError(
            "Threshold selection requires positive training outcomes."
        )

    candidate_thresholds = (
        np.sort(
            np.unique(
                positive_probabilities
            )
        )[::-1]
    )

    selected_threshold = None

    for threshold in candidate_thresholds:
        achieved_sensitivity = float(
            np.mean(
                positive_probabilities
                >= threshold
            )
        )

        if (
            achieved_sensitivity
            >= target_sensitivity
        ):
            selected_threshold = float(
                threshold
            )
            break

    if selected_threshold is None:
        selected_threshold = float(
            np.min(
                positive_probabilities
            )
        )

    predicted = (
        p >= selected_threshold
    ).astype(int)

    counts = (
        binary_classification_counts(
            y,
            predicted,
        )
    )

    sensitivity = safe_ratio(
        counts["true_positive"],
        (
            counts["true_positive"]
            + counts["false_negative"]
        ),
    )

    specificity = safe_ratio(
        counts["true_negative"],
        (
            counts["true_negative"]
            + counts["false_positive"]
        ),
    )

    return {
        "threshold": (
            selected_threshold
        ),
        "achieved_sensitivity": (
            sensitivity
        ),
        "achieved_specificity": (
            specificity
        ),
        "positive_prediction_rate": float(
            predicted.mean()
        ),
        **counts,
    }


def inner_oof_probabilities(
    outer_training_data: pd.DataFrame,
    inner_splits,
    model_template,
    target: str,
) -> np.ndarray:
    prediction = np.full(
        len(outer_training_data),
        np.nan,
        dtype=float,
    )

    X_outer = prepare_predictor_matrix(
        outer_training_data
    )

    y_outer = (
        outer_training_data[
            target
        ].to_numpy(
            dtype=int
        )
    )

    for (
        inner_training_indices,
        inner_validation_indices,
    ) in inner_splits:
        estimator = fresh_estimator(
            model_template
        )

        estimator.fit(
            X_outer.iloc[
                inner_training_indices
            ],
            y_outer[
                inner_training_indices
            ],
        )

        prediction[
            inner_validation_indices
        ] = estimator.predict_proba(
            X_outer.iloc[
                inner_validation_indices
            ]
        )[:, 1]

    if np.isnan(
        prediction
    ).any():
        raise ValueError(
            "Inner out-of-fold threshold-training probabilities are missing."
        )

    if not (
        (
            prediction >= 0
        )
        & (
            prediction <= 1
        )
    ).all():
        raise ValueError(
            "Inner out-of-fold probabilities lie outside [0, 1]."
        )

    return prediction


## 17. Nested fold-specific threshold selection and held-out application

This is the computationally intensive section. It fits 60 inner models:

$$
5\ \text{outer folds}
\times
3\ \text{inner folds}
\times
2\ \text{models}
\times
2\ \text{targets}
=
60.
$$

In [ ]:
threshold_selection_rows = []
thresholded_prediction_rows = []

threshold_start_time = time.time()
completed_inner_model_sets = 0
total_inner_model_sets = (
    N_OUTER_SPLITS
    * len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
)

for outer_fold in range(
    1,
    N_OUTER_SPLITS + 1,
):
    outer_training_data = (
        analysis
        .loc[
            analysis["cv_fold"]
            != outer_fold
        ]
        .reset_index(drop=True)
    )

    outer_validation_data = (
        analysis
        .loc[
            analysis["cv_fold"]
            == outer_fold
        ]
        .copy()
    )

    inner_splitter = (
        StratifiedKFold(
            n_splits=(
                INNER_THRESHOLD_SPLITS
            ),
            shuffle=True,
            random_state=(
                RANDOM_STATE
                + outer_fold
            ),
        )
    )

    inner_splits = list(
        inner_splitter.split(
            outer_training_data[
                PREDICTOR_COLUMNS
            ],
            outer_training_data[
                "joint_label_code"
            ],
        )
    )

    for model_name in MODEL_NAMES:
        for target in TARGET_COLUMNS:
            training_probability = (
                inner_oof_probabilities(
                    outer_training_data=(
                        outer_training_data
                    ),
                    inner_splits=(
                        inner_splits
                    ),
                    model_template=(
                        model_templates[
                            model_name
                        ][target]
                    ),
                    target=target,
                )
            )

            y_training = (
                outer_training_data[
                    target
                ].to_numpy(
                    dtype=int
                )
            )

            held_out_probability = (
                outer_validation_data[
                    PROBABILITY_COLUMNS[
                        (
                            model_name,
                            target,
                        )
                    ]
                ].to_numpy(
                    dtype=float
                )
            )

            for (
                target_sensitivity
            ) in TARGET_SENSITIVITY_LEVELS:
                selection = (
                    select_threshold_for_minimum_sensitivity(
                        y_true=(
                            y_training
                        ),
                        probability=(
                            training_probability
                        ),
                        target_sensitivity=(
                            target_sensitivity
                        ),
                    )
                )

                threshold = (
                    selection[
                        "threshold"
                    ]
                )

                held_out_class = (
                    held_out_probability
                    >= threshold
                ).astype(int)

                threshold_selection_rows.append(
                    {
                        "outer_fold": (
                            outer_fold
                        ),
                        "model": (
                            model_name
                        ),
                        "model_display_name": (
                            MODEL_DISPLAY_NAMES[
                                model_name
                            ]
                        ),
                        "target": target,
                        "target_display_name": (
                            TARGET_DISPLAY_NAMES[
                                target
                            ]
                        ),
                        "target_training_sensitivity": float(
                            target_sensitivity
                        ),
                        "selected_threshold": float(
                            threshold
                        ),
                        "outer_training_n": int(
                            len(
                                outer_training_data
                            )
                        ),
                        "outer_training_positive_n": int(
                            y_training.sum()
                        ),
                        "inner_threshold_splits": (
                            INNER_THRESHOLD_SPLITS
                        ),
                        "inner_oof_achieved_sensitivity": (
                            selection[
                                "achieved_sensitivity"
                            ]
                        ),
                        "inner_oof_achieved_specificity": (
                            selection[
                                "achieved_specificity"
                            ]
                        ),
                        "inner_oof_positive_prediction_rate": (
                            selection[
                                "positive_prediction_rate"
                            ]
                        ),
                        "held_out_n": int(
                            len(
                                outer_validation_data
                            )
                        ),
                        "held_out_positive_n": int(
                            outer_validation_data[
                                target
                            ].sum()
                        ),
                    }
                )

                for row_index, (
                    participant_id,
                    observed_outcome,
                    probability,
                    predicted_class,
                ) in enumerate(
                    zip(
                        outer_validation_data[
                            "id"
                        ].astype(int),
                        outer_validation_data[
                            target
                        ].astype(int),
                        held_out_probability,
                        held_out_class,
                    )
                ):
                    thresholded_prediction_rows.append(
                        {
                            "id": int(
                                participant_id
                            ),
                            "cv_fold": (
                                outer_fold
                            ),
                            "joint_label_code": str(
                                outer_validation_data[
                                    "joint_label_code"
                                ].iloc[
                                    row_index
                                ]
                            ),
                            "model": (
                                model_name
                            ),
                            "model_display_name": (
                                MODEL_DISPLAY_NAMES[
                                    model_name
                                ]
                            ),
                            "target": (
                                target
                            ),
                            "target_display_name": (
                                TARGET_DISPLAY_NAMES[
                                    target
                                ]
                            ),
                            "observed_outcome": int(
                                observed_outcome
                            ),
                            "probability": float(
                                probability
                            ),
                            "target_training_sensitivity": float(
                                target_sensitivity
                            ),
                            "selected_threshold": float(
                                threshold
                            ),
                            "predicted_class": int(
                                predicted_class
                            ),
                        }
                    )

            completed_inner_model_sets += 1

            elapsed_minutes = (
                time.time()
                - threshold_start_time
            ) / 60

            print(
                f"Completed {completed_inner_model_sets}/"
                f"{total_inner_model_sets} outer-fold model–target "
                f"threshold fits "
                f"({elapsed_minutes:.1f} minutes elapsed)."
            )

threshold_selection = pd.DataFrame(
    threshold_selection_rows
)

thresholded_oof_long = pd.DataFrame(
    thresholded_prediction_rows
)

expected_threshold_selection_rows = (
    N_OUTER_SPLITS
    * len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)

if (
    len(threshold_selection)
    != expected_threshold_selection_rows
):
    raise ValueError(
        "Unexpected number of fold-specific threshold rows."
    )

expected_thresholded_rows = (
    len(analysis)
    * len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)

if (
    len(thresholded_oof_long)
    != expected_thresholded_rows
):
    raise ValueError(
        "Unexpected number of thresholded participant rows."
    )

rows_per_participant = (
    thresholded_oof_long
    .groupby("id")
    .size()
)

expected_rows_per_participant = (
    len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
    * len(TARGET_SENSITIVITY_LEVELS)
)

if not rows_per_participant.eq(
    expected_rows_per_participant
).all():
    raise ValueError(
        "At least one participant lacks a complete set of thresholded predictions."
    )

threshold_selection.to_csv(
    TABLE_DIR
    / "fold_specific_threshold_selection.csv",
    index=False,
)

thresholded_oof_long.to_csv(
    THRESHOLDED_LONG_PATH,
    index=False,
)

print("Saved fold-specific thresholds and long predictions.")


## 18. Thresholded out-of-fold performance

In [ ]:
def thresholded_metrics(
    y_true: np.ndarray,
    predicted_class: np.ndarray,
) -> dict:
    y = np.asarray(
        y_true,
        dtype=int,
    )

    predicted = np.asarray(
        predicted_class,
        dtype=int,
    )

    counts = (
        binary_classification_counts(
            y,
            predicted,
        )
    )

    sensitivity = safe_ratio(
        counts["true_positive"],
        (
            counts["true_positive"]
            + counts["false_negative"]
        ),
    )

    specificity = safe_ratio(
        counts["true_negative"],
        (
            counts["true_negative"]
            + counts["false_positive"]
        ),
    )

    false_negative_rate = (
        1 - sensitivity
        if not pd.isna(
            sensitivity
        )
        else np.nan
    )

    false_positive_rate = (
        1 - specificity
        if not pd.isna(
            specificity
        )
        else np.nan
    )

    positive_predictive_value = safe_ratio(
        counts["true_positive"],
        (
            counts["true_positive"]
            + counts["false_positive"]
        ),
    )

    negative_predictive_value = safe_ratio(
        counts["true_negative"],
        (
            counts["true_negative"]
            + counts["false_negative"]
        ),
    )

    return {
        **counts,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "false_negative_rate": (
            false_negative_rate
        ),
        "false_positive_rate": (
            false_positive_rate
        ),
        "positive_prediction_rate": float(
            predicted.mean()
        ),
        "positive_predictive_value": (
            positive_predictive_value
        ),
        "negative_predictive_value": (
            negative_predictive_value
        ),
    }


threshold_performance_rows = []

for (
    model_name,
    target,
    target_sensitivity,
), group in thresholded_oof_long.groupby(
    [
        "model",
        "target",
        "target_training_sensitivity",
    ],
    observed=True,
):
    metrics = thresholded_metrics(
        group[
            "observed_outcome"
        ].to_numpy(
            dtype=int
        ),
        group[
            "predicted_class"
        ].to_numpy(
            dtype=int
        ),
    )

    threshold_performance_rows.append(
        {
            "model": model_name,
            "model_display_name": (
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
            ),
            "target": target,
            "target_display_name": (
                TARGET_DISPLAY_NAMES[
                    target
                ]
            ),
            "target_training_sensitivity": float(
                target_sensitivity
            ),
            "evaluation_n": int(
                len(group)
            ),
            "positive_n": int(
                group[
                    "observed_outcome"
                ].sum()
            ),
            **metrics,
        }
    )

thresholded_performance = pd.DataFrame(
    threshold_performance_rows
)

thresholded_performance.to_csv(
    TABLE_DIR
    / "thresholded_oof_performance.csv",
    index=False,
)

thresholded_performance


### Why held-out sensitivity is not exactly 70%, 80%, or 90%

The sensitivity target is applied to inner out-of-fold predictions from the outer training sample. The resulting threshold is then transferred unchanged to the held-out fold.

Consequently, held-out sensitivity can differ from the requested training value. This is expected and is evidence that the threshold was not selected from held-out outcomes.

## 19. Threshold distributions across outer folds

In [ ]:
threshold_distribution = (
    threshold_selection
    .groupby(
        [
            "model",
            "target",
            "target_training_sensitivity",
        ],
        as_index=False,
    )
    .agg(
        mean_selected_threshold=(
            "selected_threshold",
            "mean",
        ),
        standard_deviation_selected_threshold=(
            "selected_threshold",
            "std",
        ),
        minimum_selected_threshold=(
            "selected_threshold",
            "min",
        ),
        maximum_selected_threshold=(
            "selected_threshold",
            "max",
        ),
        mean_inner_achieved_sensitivity=(
            "inner_oof_achieved_sensitivity",
            "mean",
        ),
        mean_inner_achieved_specificity=(
            "inner_oof_achieved_specificity",
            "mean",
        ),
    )
)

threshold_distribution[
    "model_display_name"
] = (
    threshold_distribution[
        "model"
    ].map(
        MODEL_DISPLAY_NAMES
    )
)

threshold_distribution[
    "target_display_name"
] = (
    threshold_distribution[
        "target"
    ].map(
        TARGET_DISPLAY_NAMES
    )
)

threshold_distribution.to_csv(
    TABLE_DIR
    / "threshold_distribution_across_folds.csv",
    index=False,
)

threshold_distribution


## 20. Save wide and primary-threshold files for Notebook 07

In [ ]:
thresholded_oof_long[
    "prediction_key"
] = (
    thresholded_oof_long[
        "model"
    ]
    + "__"
    + thresholded_oof_long[
        "target"
    ]
    + "__sens_"
    + (
        100
        * thresholded_oof_long[
            "target_training_sensitivity"
        ]
    )
    .round()
    .astype(int)
    .astype(str)
)

wide_index_columns = [
    "id",
    "cv_fold",
    "joint_label_code",
]

wide_probability = (
    thresholded_oof_long
    .pivot(
        index=wide_index_columns,
        columns="prediction_key",
        values="probability",
    )
    .add_prefix("probability__")
)

wide_threshold = (
    thresholded_oof_long
    .pivot(
        index=wide_index_columns,
        columns="prediction_key",
        values="selected_threshold",
    )
    .add_prefix("threshold__")
)

wide_class = (
    thresholded_oof_long
    .pivot(
        index=wide_index_columns,
        columns="prediction_key",
        values="predicted_class",
    )
    .add_prefix("predicted_class__")
)

thresholded_oof_wide = (
    pd.concat(
        [
            wide_probability,
            wide_threshold,
            wide_class,
        ],
        axis=1,
    )
    .reset_index()
)

thresholded_oof_wide.columns.name = None

target_values = analysis[
    [
        "id",
        *TARGET_COLUMNS,
    ]
].copy()

thresholded_oof_wide = (
    target_values
    .merge(
        thresholded_oof_wide,
        on="id",
        how="left",
        validate="one_to_one",
    )
)

thresholded_oof_wide.to_csv(
    THRESHOLDED_WIDE_PATH,
    index=False,
)

primary_threshold_predictions = (
    thresholded_oof_long
    .loc[
        np.isclose(
            thresholded_oof_long[
                "target_training_sensitivity"
            ],
            PRIMARY_TARGET_SENSITIVITY,
        )
    ]
    .drop(
        columns=[
            "prediction_key",
        ]
    )
    .copy()
)

expected_primary_rows = (
    len(analysis)
    * len(MODEL_NAMES)
    * len(TARGET_COLUMNS)
)

if (
    len(
        primary_threshold_predictions
    )
    != expected_primary_rows
):
    raise ValueError(
        "The primary 80%-sensitivity prediction file has an unexpected size."
    )

primary_threshold_predictions.to_csv(
    PRIMARY_THRESHOLD_PATH,
    index=False,
)

print("Saved:")
print(THRESHOLDED_LONG_PATH)
print(THRESHOLDED_WIDE_PATH)
print(PRIMARY_THRESHOLD_PATH)


## 21. Plot held-out sensitivity and specificity across operating points

In [ ]:
threshold_figure_paths = {}

for target in TARGET_COLUMNS:
    target_data = (
        thresholded_performance
        .loc[
            thresholded_performance[
                "target"
            ]
            == target
        ]
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(7.5, 5.5)
    )

    for model_name in MODEL_NAMES:
        model_data = (
            target_data
            .loc[
                target_data[
                    "model"
                ]
                == model_name
            ]
            .sort_values(
                "target_training_sensitivity"
            )
        )

        axis.plot(
            100
            * model_data[
                "target_training_sensitivity"
            ],
            100
            * model_data[
                "sensitivity"
            ],
            marker="o",
            label=(
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
                + " — held-out sensitivity"
            ),
        )

        axis.plot(
            100
            * model_data[
                "target_training_sensitivity"
            ],
            100
            * model_data[
                "specificity"
            ],
            marker="s",
            linestyle="--",
            label=(
                MODEL_DISPLAY_NAMES[
                    model_name
                ]
                + " — held-out specificity"
            ),
        )

    axis.set_xlabel(
        "Target sensitivity used for training-fold threshold selection (%)"
    )

    axis.set_ylabel(
        "Held-out metric (%)"
    )

    axis.set_title(
        "Threshold operating points: "
        f"{TARGET_SHORT_NAMES[target]}"
    )

    axis.set_xticks(
        [
            70,
            80,
            90,
        ]
    )

    axis.set_ylim(
        0,
        100,
    )

    axis.legend()
    figure.tight_layout()

    filename_target = (
        "prior_diagnosis"
        if target
        == "self_reported_prior_diagnosis"
        else "hba1c_ge_6_5"
    )

    png_path = (
        FIGURE_DIR
        / (
            "threshold_operating_points_"
            f"{filename_target}.png"
        )
    )

    pdf_path = (
        FIGURE_DIR
        / (
            "threshold_operating_points_"
            f"{filename_target}.pdf"
        )
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    threshold_figure_paths[
        f"{filename_target}_png"
    ] = png_path

    threshold_figure_paths[
        f"{filename_target}_pdf"
    ] = pdf_path

threshold_figure_paths


## Final outputs and reproducibility metadata

In [ ]:
def json_safe(value):
    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        dict,
    ):
        return {
            str(key): json_safe(item)
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


output_files = {
    "performance_point_estimates": (
        TABLE_DIR
        / "performance_point_estimates_long.csv"
    ),
    "performance_bootstrap_long": (
        TABLE_DIR
        / "performance_bootstrap_metrics_long.csv"
    ),
    "performance_summary": (
        TABLE_DIR
        / "performance_metrics_with_bootstrap_intervals.csv"
    ),
    "table3_long": (
        TABLE_DIR
        / "table3_primary_predictive_performance_long.csv"
    ),
    "table3_wide": (
        TABLE_DIR
        / "table3_primary_predictive_performance_wide.csv"
    ),
    "within_target_model_differences": (
        TABLE_DIR
        / "performance_within_target_ebm_minus_logistic.csv"
    ),
    "cross_target_differences": (
        TABLE_DIR
        / "performance_cross_target_hba1c_minus_prior.csv"
    ),
    "calibration_points": (
        TABLE_DIR
        / "calibration_curve_points.csv"
    ),
    "fold_specific_thresholds": (
        TABLE_DIR
        / "fold_specific_threshold_selection.csv"
    ),
    "thresholded_performance": (
        TABLE_DIR
        / "thresholded_oof_performance.csv"
    ),
    "threshold_distribution": (
        TABLE_DIR
        / "threshold_distribution_across_folds.csv"
    ),
    "thresholded_predictions_long": (
        THRESHOLDED_LONG_PATH
    ),
    "thresholded_predictions_wide": (
        THRESHOLDED_WIDE_PATH
    ),
    "primary_80_sensitivity_predictions": (
        PRIMARY_THRESHOLD_PATH
    ),
    **{
        f"calibration_{name}": path
        for name, path
        in calibration_figure_paths.items()
    },
    **{
        f"threshold_{name}": path
        for name, path
        in threshold_figure_paths.items()
    },
}

missing_output_files = [
    str(path)
    for path in output_files.values()
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "The following expected Notebook 06 outputs are missing:\n"
        + "\n".join(
            missing_output_files
        )
    )

performance_metadata = {
    "analytic_sample_n": int(
        len(analysis)
    ),
    "analysis_id_sha256": (
        analysis_id_hash
    ),
    "random_state": RANDOM_STATE,
    "outer_folds": (
        N_OUTER_SPLITS
    ),
    "inner_threshold_selection_folds": (
        INNER_THRESHOLD_SPLITS
    ),
    "models": MODEL_NAMES,
    "targets": TARGET_COLUMNS,
    "probability_columns": {
        (
            model_name
            + "__"
            + target
        ): probability_column
        for (
            model_name,
            target
        ), probability_column
        in PROBABILITY_COLUMNS.items()
    },
    "performance_bootstrap": {
        "replicates": (
            N_BOOTSTRAP
        ),
        "resampling_unit": (
            "participant row"
        ),
        "paired_across_models": True,
        "paired_across_targets": True,
        "models_refit_inside_bootstrap": False,
        "confidence_level": (
            CONFIDENCE_LEVEL
        ),
        "interval_method": (
            "percentile"
        ),
    },
    "metrics": {
        "roc_auc": True,
        "pr_auc": True,
        "target_prevalence": True,
        "brier_score": True,
        "brier_skill_score": (
            "1 - model Brier / constant-prevalence Brier"
        ),
        "calibration_intercept_slope": (
            "Joint logistic recalibration model fitted to logit probabilities"
        ),
        "sensitivity_specificity_at_0_5": True,
    },
    "threshold_selection": {
        "method": (
            "Nested inner out-of-fold predictions within each outer training set"
        ),
        "target_sensitivities": (
            TARGET_SENSITIVITY_LEVELS
        ),
        "primary_target_sensitivity": (
            PRIMARY_TARGET_SENSITIVITY
        ),
        "selection_rule": (
            "Highest threshold with inner-OOF sensitivity "
            "at least the requested value"
        ),
        "held_out_outcomes_used_for_selection": False,
        "target_specific_thresholds": True,
        "model_specific_thresholds": True,
        "fold_specific_thresholds": True,
    },
    "software_versions": {
        "python": (
            platform.python_version()
        ),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": (
            sklearn.__version__
        ),
        "scipy": __import__(
            "scipy"
        ).__version__,
        "interpret": (
            interpret.__version__
        ),
        "joblib": (
            joblib.__version__
        ),
    },
    "output_files": {
        name: str(path)
        for name, path
        in output_files.items()
    },
}

with PERFORMANCE_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        json_safe(
            performance_metadata
        ),
        file,
        indent=2,
    )

final_checkpoint = {
    "analytic_sample_n": int(
        len(analysis)
    ),
    "same_participants_as_notebook04": bool(
        logistic_metadata[
            "analysis_id_sha256"
        ]
        == analysis_id_hash
    ),
    "same_participants_as_notebook05": bool(
        ebm_metadata[
            "analysis_id_sha256"
        ]
        == analysis_id_hash
    ),
    "four_model_target_combinations": bool(
        performance_point_estimates[
            [
                "model",
                "target",
            ]
        ]
        .drop_duplicates()
        .shape[0]
        == 4
    ),
    "bootstrap_replicates_completed": int(
        bootstrap_performance[
            "bootstrap_replicate"
        ].nunique()
    ),
    "table3_rows": int(
        len(
            table3_wide
        )
    ),
    "calibration_curves_saved": bool(
        len(
            calibration_figure_paths
        )
        == 4
    ),
    "fold_specific_threshold_rows": int(
        len(
            threshold_selection
        )
    ),
    "thresholded_prediction_rows": int(
        len(
            thresholded_oof_long
        )
    ),
    "primary_80_sensitivity_rows": int(
        len(
            primary_threshold_predictions
        )
    ),
    "held_out_outcomes_used_for_threshold_selection": False,
    "all_expected_outputs_saved": bool(
        len(
            missing_output_files
        )
        == 0
    ),
    "metadata_saved": bool(
        PERFORMANCE_METADATA_PATH.exists()
    ),
}

print("Saved Notebook 06 metadata to:")
print(PERFORMANCE_METADATA_PATH)
print()
print("Final checkpoint:")
final_checkpoint


## Completion criteria

- Performance is evaluated only on out-of-fold probabilities.
- Threshold selection uses outer-training data and never the held-out outcomes.
- All performance, threshold, and metadata files are written.